> **Public-release version:** Private animal-level inputs and source-derived observations have been removed from the notebook. The original modelling and plotting cells are retained. See the repository README for data requirements.

# NB15 — GI hierarchical Bayesian donor abundance and population composition

**Purpose.** Test the hypothesis that the **underlying mean recovered donor (D) abundance is shared between target and non-target animals at each sampling day**, while estimating between-animal donor variability from the selective-culture measurements. This is an **exploratory** model and does not modify the manuscript or earlier notebooks.

**Study and endpoints.** The GI confirmation experiment has **16 animals** (8 target, 8 non-target), each sampled on **Days 10 and 14** (32 animal-day records). Animals received the same *administered* donor inoculum on Days 7, 8 and 9; equal dosing **does not establish equal recovered CFU/g**. The same animals contribute observations on both days; no measured initial R:D mixture or complete time trajectory is available.

**Latent populations and observed selective plates (CFU/g):**

| Latent population | Biological definition | Main plated-count relation |
|---|---|---|
| **R** | Recipients without Td | Sm = R + D + T2 |
| **D** | Original Td-bearing donor cells | SmTc = D + T2 |
| **T1** | Transconjugants retaining carbenicillin resistance | SmCbTc = T1 |
| **Ts = T2−T1** | Td-containing transconjugants that are carbenicillin-sensitive | SmRfTc = T2 = T1 + Ts |

We estimate four **mutually exclusive** fractions **R, D, T1, Ts**, normalized for each animal in each posterior draw by **R + D + T2**; the four fractions sum to 100%. **T1 is a subset of T2; do not plot T2 as a separate fifth segment.** Observed counts with inconsistent plate-subset relationships are retained as observed; no clipping or subtraction of negative observed components.

**Hierarchical donor model.** For day \(t\) (10 or 14) and animal \(i\):

\[
\mu_{D,t}\sim\operatorname{Normal}(m_0,s_0^2),\quad
\sigma_D\sim\operatorname{HalfNormal}(0,s_D^2),\quad
z_{D,i}\sim\operatorname{Normal}(0,1),
\]
\[
\log D_{i,t}=\mu_{D,t}+\sigma_D z_{D,i} + \tau_{D,visit}z_{i,t,D},\quad
z_{i,t,D}\sim\operatorname{Normal}(0,1).
\]

Thus **\(\sigma_D\) is learned from data** and describes persistent donor-specific between-animal variation. The additional within-animal visit scale **\(\tau_{D,visit}=1.8\)** is held fixed, as in NB14, to distinguish visit-to-visit variability; this assumed scale can limit identification of \(\sigma_D\). The donor mean \(\mu_{D,t}\) is **shared across target status within day**, but its values may differ on Days 10 and 14. Other populations have group-specific log-abundance locations and an animal-level random effect as in NB14. Separate robust Student-t observation distributions capture noise in each plate measurement.

**Four sensitivity scenarios (same underlying biology and count data):**

1. **Baseline:** \(m_0=12\), \(s_0=8\), \(s_D=2.5\).
2. **Prior location:** \(m_0=18\), \(s_0=8\), \(s_D=2.5\).
3. **Prior spread:** \(m_0=12\), \(s_0=5\), \(s_D=2.5\).
4. **Donor SD prior:** \(m_0=12\), \(s_0=8\), \(s_D=1.0\).

Prior values are on the **natural log(CFU/g)** scale, not the initial administered CFU dose. Changing these assumptions tests prior dependence; none is experimentally established. **Plate log-error SD = 0.7**, non-donor animal scale **0.9**, and visit-level scales **1.8** remain fixed exploratory assumptions. Two zero T1 counts are **left-censored** using an *unverified surrogate* colony-detection limit. Posterior convergence, plate predictive checks and held-out plate checks must be reviewed before biological interpretation.

**Outputs.** 100%-stacked four-part bar plot (baseline), uncertainty and prior-sensitivity figures; posterior fractions for all four groups and animals; inferred absolute donor abundance, \(\sigma_D\), convergence diagnostics, plate checks, and group contrasts. **The shared-donor model assumes equal group-level donor means; its fit alone does not prove this assumption.** The data cannot identify Td transfer rates, growth rates or molecular mechanism.

**Input:** The original KWS-derived data (32 animal-day records) are not included. Separately authorised input must be supplied in the working data directory. Results are saved in `Transmid_Animal_Bayesian_Project/06_Results/NB15_GI_Hierarchical_Donor` (or local Colab storage if Drive is unavailable). KWS-derived data are private and must not be published without permission.

**Instructions:** Run the numbered, collapsible cells sequentially in Google Colab. Do **not** interpret results if convergence fails. NB05, NB08, NB13, NB14, the manuscript, supplement and GitHub repository remain unchanged.

In [ ]:
#@title NB15.01 — Install Python analysis packages
import sys,subprocess
subprocess.check_call([sys.executable,'-m','pip','install','-q','cmdstanpy','numpy','pandas','matplotlib','scipy'])
print('NB14 dependencies installed.')


In [ ]:
#@title NB15.02 — Locate existing Google Drive project and create isolated output folder
from pathlib import Path
PROJECT=None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    root=Path('/content/drive/MyDrive')
    direct=root/'Transmid_Animal_Bayesian_Project'
    if direct.is_dir():
        PROJECT=direct
    else:
        hits=[p for p in root.rglob('Transmid_Animal_Bayesian_Project') if p.is_dir()]
        if len(hits)==1: PROJECT=hits[0]
        elif len(hits)>1: raise RuntimeError(f'Multiple project folders found: {hits}')
except ImportError:
    print('Colab Drive interface unavailable. Using local notebook storage.')
if PROJECT is None:
    PROJECT=Path('/content/NB15_GI_Project')
    print('No unique existing project found; using Colab local storage:',PROJECT)
RESULT_DIR=PROJECT/'06_Results'/'NB15_GI_Hierarchical_Donor'
MODEL_DIR=PROJECT/'05_Stan_Models'
DATA_DIR=PROJECT/'03_Analysis_Data'/'NB15_GI'
for directory in [RESULT_DIR,MODEL_DIR,DATA_DIR]: directory.mkdir(parents=True,exist_ok=True)
print('Output:',RESULT_DIR)


In [ ]:
#@title NB15.03 — Load separately authorised GI culture-count data
# Public-release version: original animal-day culture counts are NOT included.
# Place an authorised copy of NB15_GI_original_selective_plate_CFU_g.csv in DATA_DIR.
import numpy as np, pandas as pd
DATA_NAME='NB15_GI_original_selective_plate_CFU_g.csv'
INPUT=DATA_DIR/DATA_NAME
if not INPUT.is_file():
    raise FileNotFoundError(f'Private GI input not provided: {INPUT}. See README for required columns.')
data=pd.read_csv(INPUT,dtype={'animal_id':str})
PLATES=['N_Sm','N_SmTc','N_SmRfTc','N_SmCbTc']
HELDOUT=['N_SmRf','N_SmCb']
GROUPS=['Day 10 — Target','Day 10 — Non-target','Day 14 — Target','Day 14 — Non-target']
CATS=['R','D','T1','T_sensitive']
required={'animal_id','day','group','sample_weight_g',*PLATES,*HELDOUT}
assert required.issubset(data.columns), f'Missing GI columns: {required-set(data.columns)}'
assert len(data)==32 and data.animal_id.nunique()==16
assert set(data.day)=={10,14} and set(data.group)=={'Target','Non-target'}
assert data.groupby(['day','group']).size().eq(8).all()
assert data.groupby('animal_id').day.nunique().eq(2).all()
assert data.groupby('animal_id').group.nunique().eq(1).all()
assert np.isfinite(data[PLATES+HELDOUT+['sample_weight_g']].to_numpy(float)).all()
assert (data[PLATES+HELDOUT]>=0).all().all() and (data.sample_weight_g>0).all()
assert (data[['N_Sm','N_SmTc','N_SmRfTc']]>0).all().all()
assert (data.N_SmCbTc==0).sum()==2
data['group_day']=data.day.map(lambda d:f'Day {d}')+' — '+data.group
data['animal_index']=data.animal_id.map({a:i+1 for i,a in enumerate(sorted(data.animal_id.unique()))})
data['group_index']=data.group_day.map({g:i+1 for i,g in enumerate(GROUPS)})
data['day_index']=data.day.map({10:1,14:2})
assert (data[['group_index','animal_index','day_index']]>0).all().all()
print('GI input validated; 32 records, 16 mice (private source supplied separately).')


In [ ]:
#@title NB15.04 — Audit original counts and donor proxies without truncation
import numpy as np,pandas as pd
AUDIT=data[['animal_id','day','group','group_day','sample_weight_g',*PLATES,*HELDOUT]].copy()
AUDIT['raw_D_proxy_CFU_g']=AUDIT.N_SmTc-AUDIT.N_SmRfTc
AUDIT['raw_R_proxy_CFU_g']=AUDIT.N_Sm-AUDIT.N_SmTc
AUDIT['raw_T2_CFU_g']=AUDIT.N_SmRfTc
AUDIT['raw_T1_CFU_g']=AUDIT.N_SmCbTc
AUDIT['D_proxy_negative']=AUDIT.raw_D_proxy_CFU_g<0
AUDIT['R_proxy_negative']=AUDIT.raw_R_proxy_CFU_g<0
AUDIT['T1_gt_T2']=AUDIT.raw_T1_CFU_g>AUDIT.raw_T2_CFU_g
AUDIT.to_csv(RESULT_DIR/'NB15_original_count_audit.csv',index=False)
# Only nonnegative donor proxies can be interpreted as empirical donor abundance;
# this is an illustrative proxy, not an independent donor-specific plate count.
summary=AUDIT.groupby('group_day',sort=False).agg(n=('animal_id','size'),
           donor_proxy_median=('raw_D_proxy_CFU_g','median'),
           donor_proxy_negative=('D_proxy_negative','sum'),
           R_proxy_negative=('R_proxy_negative','sum'),T1_gt_T2=('T1_gt_T2','sum'))
print('Direct plate-derived donor proxy (SmTc minus SmRfTc), CFU/g:')
display(summary.reindex(GROUPS))
print('Discrepancies remain in the model as noisy observations. No clipping.')


In [ ]:
#@title NB15.05 — Define and compile hierarchical shared-donor Stan model
from cmdstanpy import CmdStanModel, cmdstan_path, install_cmdstan
import tempfile
try:
    print('CmdStan available:',cmdstan_path())
except ValueError:
    print('Installing CmdStan in the current Colab runtime.')
    if not install_cmdstan(cores=2,progress=True):
        raise RuntimeError('CmdStan installation failed.')

PLATE_LOG_SD=np.array([0.7,0.7,0.7,0.7])
OCCASION_LOG_SD=np.array([1.8,1.8,1.8,1.8])
NON_D_ANIMAL_LOG_SD=0.9
T1_CFU_ML_SURROGATE_LIMIT=20.0
T1_LOD=T1_CFU_ML_SURROGATE_LIMIT/data.sample_weight_g.to_numpy(float)

STAN_SOURCE=r'''
// NB15 — hierarchical GI donor abundance, shared mean by day.
// Latent logpop columns: R, D, T1, Ts where Ts=T2−T1.
// Donor-specific animal effects persist across Day 10 and 14.
// Fixed visit variability and plate-error SDs are exploratory assumptions.
data {
  int<lower=1> N;
  int<lower=1> G;
  int<lower=1> A;
  array[N] int<lower=1,upper=G> group;
  array[N] int<lower=1,upper=A> animal;
  array[N] int<lower=1,upper=2> day;
  matrix<lower=0>[N,4] y;
  vector<lower=0>[N] lod_T1;
  vector<lower=0>[4] sd_plate;
  vector<lower=0>[4] sd_occasion;
  real<lower=0> sd_nondonor_animal;
  real prior_D_center;
  real<lower=0> prior_D_sd;
  real<lower=0> prior_sigma_D_scale;
}
parameters {
  matrix[G,3] mu_other_log;  // R, T1, Ts (group-specific)
  vector[2] mu_D_log;         // donor mean: day-specific and shared target/non-target
  real<lower=0> sigma_D;      // donor-specific between-animal SD; learned
  vector[A] donor_z;         // same donor random effect at both sampling days
  vector[A] animal_z;        // shared-animal effects for R, T1, Ts
  matrix[N,4] visit_z;       // within-animal visit deviations
}
transformed parameters {
  matrix[N,4] logpop;
  for (i in 1:N) {
    real common_nondonor = sd_nondonor_animal * animal_z[animal[i]];
    logpop[i,1] = mu_other_log[group[i],1]
                  + common_nondonor + sd_occasion[1]*visit_z[i,1];
    logpop[i,2] = mu_D_log[day[i]]
                  + sigma_D * donor_z[animal[i]]
                  + sd_occasion[2]*visit_z[i,2];
    logpop[i,3] = mu_other_log[group[i],2]
                  + common_nondonor + sd_occasion[3]*visit_z[i,3];
    logpop[i,4] = mu_other_log[group[i],3]
                  + common_nondonor + sd_occasion[4]*visit_z[i,4];
  }
}
model {
  to_vector(mu_other_log) ~ normal(12,8);
  mu_D_log ~ normal(prior_D_center,prior_D_sd);
  sigma_D ~ normal(0,prior_sigma_D_scale); // half-Normal by lower=0 constraint
  donor_z ~ std_normal();
  animal_z ~ std_normal();
  to_vector(visit_z) ~ std_normal();
  for (i in 1:N) {
    real log_T2 = log_sum_exp(logpop[i,3], logpop[i,4]);
    real log_SmTc = log_sum_exp(logpop[i,2], log_T2);
    real log_Sm = log_sum_exp(logpop[i,1], log_SmTc);
    log(y[i,1]) ~ student_t(4,log_Sm,sd_plate[1]);
    log(y[i,2]) ~ student_t(4,log_SmTc,sd_plate[2]);
    log(y[i,3]) ~ student_t(4,log_T2,sd_plate[3]);
    if (y[i,4] > 0)
      log(y[i,4]) ~ student_t(4,logpop[i,3],sd_plate[4]);
    else
      target += student_t_lcdf(log(lod_T1[i]) | 4,logpop[i,3],sd_plate[4]);
  }
}
generated quantities {
  matrix[N,4] log_y_rep;
  for (i in 1:N) {
    real log_T2 = log_sum_exp(logpop[i,3], logpop[i,4]);
    real log_SmTc = log_sum_exp(logpop[i,2], log_T2);
    real log_Sm = log_sum_exp(logpop[i,1], log_SmTc);
    log_y_rep[i,1] = student_t_rng(4,log_Sm,sd_plate[1]);
    log_y_rep[i,2] = student_t_rng(4,log_SmTc,sd_plate[2]);
    log_y_rep[i,3] = student_t_rng(4,log_T2,sd_plate[3]);
    log_y_rep[i,4] = student_t_rng(4,logpop[i,3],sd_plate[4]);
  }
}
'''

STAN_FILE_CLOUD=MODEL_DIR/'NB15_GI_hierarchical_shared_donor.stan'
if STAN_FILE_CLOUD.exists():
    if STAN_FILE_CLOUD.read_text()!=STAN_SOURCE:
        raise ValueError('An existing NB15 Stan model differs; refusing to overwrite.')
else:
    STAN_FILE_CLOUD.write_text(STAN_SOURCE)
RUNTIME_DIR=Path(tempfile.mkdtemp(prefix='nb15_gi_'))
STAN_FILE=RUNTIME_DIR/STAN_FILE_CLOUD.name
STAN_FILE.write_text(STAN_SOURCE)
model=CmdStanModel(stan_file=str(STAN_FILE))
print('Compiled NB15 hierarchical model; model SHA-256:',__import__('hashlib').sha256(STAN_SOURCE.encode()).hexdigest())


In [ ]:
#@title NB15.06 — Fit shared-donor model under four prior scenarios
SCENARIOS={
  'Baseline':          {'prior_D_center':12., 'prior_D_sd':8., 'prior_sigma_D_scale':2.5},
  'Location18':        {'prior_D_center':18., 'prior_D_sd':8., 'prior_sigma_D_scale':2.5},
  'Spread5':           {'prior_D_center':12., 'prior_D_sd':5., 'prior_sigma_D_scale':2.5},
  'SigmaPrior1':       {'prior_D_center':12., 'prior_D_sd':8., 'prior_sigma_D_scale':1.0},
}
BASE={
 'N':len(data),'G':4,'A':int(data.animal_index.nunique()),
 'group':data.group_index.astype(int).tolist(),
 'animal':data.animal_index.astype(int).tolist(),
 'day':data.day_index.astype(int).tolist(),
 'y':data[PLATES].to_numpy(float),
 'lod_T1':T1_LOD.tolist(),
 'sd_plate':PLATE_LOG_SD.tolist(),
 'sd_occasion':OCCASION_LOG_SD.tolist(),
 'sd_nondonor_animal':NON_D_ANIMAL_LOG_SD
}
import json
(RESULT_DIR/'NB15_scenarios.json').write_text(json.dumps(SCENARIOS,indent=2))
fits={}
for j,(name,priors) in enumerate(SCENARIOS.items()):
    inp={**BASE,**priors}
    directory=RUNTIME_DIR/name
    directory.mkdir(parents=True,exist_ok=True)
    print('\nFitting:',name,'donor priors=',priors)
    fit=model.sample(data=inp,chains=4,parallel_chains=2,
        iter_warmup=2000,iter_sampling=2500,seed=20261126+j,
        adapt_delta=0.97,max_treedepth=12,
        output_dir=str(directory),show_progress=True)
    fits[name]=fit
    print(name,'sampling complete; convergence must be checked in Cell NB15.07.')


In [ ]:
#@title NB15.07 — Convergence diagnostics and validation gate
from IPython.display import display
DIAG_ROWS=[]
for name,fit in fits.items():
    sm=fit.summary()
    mask=sm.index.to_series().str.startswith((
       'mu_other_log[','mu_D_log[','sigma_D','donor_z[','animal_z[',
       'visit_z[','logpop['))
    parameters=sm.loc[mask]
    assert len(parameters)>0, 'No Stan model parameters selected for diagnostics.'
    rhat=float(parameters['R_hat'].max())
    ess=float(parameters['ESS_bulk'].min())
    mv=fit.method_variables()
    divergences=int(np.asarray(mv['divergent__']).sum())
    treedepth_hits=int(np.sum(np.asarray(mv['treedepth__'])>=12))
    passed=(np.isfinite(rhat) and rhat<1.01 and
        np.isfinite(ess) and ess>=400 and
        divergences==0 and treedepth_hits==0)
    DIAG_ROWS.append(dict(scenario=name,max_Rhat=rhat,min_bulk_ESS=ess,
        divergences=divergences,treedepth_hits=treedepth_hits,converged=bool(passed)))
    parameters[['Mean','StdDev','ESS_bulk','R_hat']].to_csv(
        RESULT_DIR/f'NB15_{name}_parameter_diagnostics.csv')
diagnostics=pd.DataFrame(DIAG_ROWS)
diagnostics.to_csv(RESULT_DIR/'NB15_convergence_summary.csv',index=False)
display(diagnostics)
if not diagnostics.converged.all():
    raise RuntimeError('At least one NB15 sensitivity fit failed convergence. Do not interpret fractions; review diagnostics before proceeding.')
print('All four scenarios passed computational convergence criteria. Biological assumptions remain exploratory.')


In [ ]:
#@title NB15.08 — Posterior donor mean, estimated donor SD and day contrast
from scipy.special import logsumexp
DONOR_ROWS=[]; DAY_ROWS=[]
for name,fit in fits.items():
    means=np.asarray(fit.stan_variable('mu_D_log'))
    sds=np.asarray(fit.stan_variable('sigma_D')).reshape(-1)
    assert means.ndim==2 and means.shape[1]==2 and means.shape[0]==sds.size
    for d,idx in [(10,0),(14,1)]:
        # exp(mu_D) is a day-specific reference donor CFU/g, not the starting dose.
        raw=np.exp(means[:,idx]);lo,med,hi=np.quantile(raw,[.025,.5,.975])
        DONOR_ROWS.append(dict(scenario=name,quantity='reference_D_CFU_g',day=d,
                               median=med,q025=lo,q975=hi))
        lo,med,hi=np.quantile(means[:,idx],[.025,.5,.975])
        DONOR_ROWS.append(dict(scenario=name,quantity='mu_D_log',day=d,
                               median=med,q025=lo,q975=hi))
    for quantity,values in [('sigma_D_log',sds),('donor_between_animal_fold_SD',np.exp(sds))]:
        lo,med,hi=np.quantile(values,[.025,.5,.975])
        DONOR_ROWS.append(dict(scenario=name,quantity=quantity,day=np.nan,
                               median=med,q025=lo,q975=hi))
    change=means[:,1]-means[:,0]
    ratio=np.exp(change)
    lo,med,hi=np.quantile(ratio,[.025,.5,.975])
    DAY_ROWS.append(dict(scenario=name,quantity='D_day14_over_day10_reference_ratio',
                         median=med,q025=lo,q975=hi,prob_greater_1=float((ratio>1).mean())))
DONOR_POST=pd.DataFrame(DONOR_ROWS)
DAY_CHANGE=pd.DataFrame(DAY_ROWS)
DONOR_POST.to_csv(RESULT_DIR/'NB15_donor_hierarchical_posterior.csv',index=False)
DAY_CHANGE.to_csv(RESULT_DIR/'NB15_donor_day_change.csv',index=False)
display(DONOR_POST.round(4))
display(DAY_CHANGE.round(4))
print('sigma_D is inferred from the plates, conditional on the fixed visit and measurement scales.')


In [ ]:
#@title NB15.09 — Normalize R, D, T1 and T2−T1 for every posterior draw
CATS=['R','D','T1','T_sensitive']
POST_FRACTIONS={}; POST_LOGPOPS={}; POST_GROUP={}; group_rows=[];animal_rows=[]
for name,fit in fits.items():
    lp=np.asarray(fit.stan_variable('logpop'))
    assert lp.ndim==3 and lp.shape[1:]==(len(data),4),lp.shape
    frac=np.exp(lp-logsumexp(lp,axis=2,keepdims=True))
    assert np.isfinite(frac).all()
    assert np.allclose(frac.sum(axis=2),1.,atol=1e-10)
    POST_FRACTIONS[name]=frac
    POST_LOGPOPS[name]=lp
    for g in GROUPS:
        ix=(data.group_day==g).to_numpy()
        draws=frac[:,ix,:].mean(axis=1)
        POST_GROUP[name,g]=draws
        assert np.allclose(draws.sum(axis=1),1.,atol=1e-10)
        for k,cat in enumerate(CATS):
            v=draws[:,k]
            lo,med,hi=np.quantile(v,[.025,.5,.975])
            group_rows.append(dict(scenario=name,group=g,component=cat,
                posterior_mean=float(v.mean()),median=float(med),q025=float(lo),q975=float(hi)))
    for i,rec in data.iterrows():
        for k,cat in enumerate(CATS):
            v=frac[:,i,k]
            lo,med,hi=np.quantile(v,[.025,.5,.975])
            animal_rows.append(dict(scenario=name,animal_id=rec.animal_id,day=int(rec.day),
                group=rec.group,component=cat,posterior_mean=float(v.mean()),
                median=float(med),q025=float(lo),q975=float(hi)))
GROUP_COMP=pd.DataFrame(group_rows)
ANIMAL_COMP=pd.DataFrame(animal_rows)
assert np.allclose(GROUP_COMP.groupby(['scenario','group']).posterior_mean.sum(),1.,atol=1e-10)
GROUP_COMP.to_csv(RESULT_DIR/'NB15_group_population_fractions.csv',index=False)
ANIMAL_COMP.to_csv(RESULT_DIR/'NB15_animal_population_fractions.csv',index=False)
print('Baseline four-part population composition, percentage (each bar sums to exactly 100%):')
display((GROUP_COMP[GROUP_COMP.scenario.eq('Baseline')].pivot(index='group',columns='component',values='posterior_mean')
         .reindex(GROUPS)[CATS]*100).round(3))


In [ ]:
#@title NB15.10 — GI population composition bar plot (baseline)
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(10,6.2))
x=np.arange(len(GROUPS))
V=(GROUP_COMP[GROUP_COMP.scenario.eq('Baseline')]
   .pivot(index='group',columns='component',values='posterior_mean')
   .reindex(GROUPS)[CATS].to_numpy())
assert np.allclose(V.sum(axis=1),1.,atol=1e-10)
long_labels=['R — recipients without Td','D — original Td donors',
             'T1 — resistant transconjugants','T2−T1 — sensitive transconjugants']
hatches=['','///','\\\\','xx']
colors=['#d5dce6','#729ac2','#eeae75','#82b4a0']
bottom=np.zeros(4)
for k in range(4):
    ax.bar(x,100*V[:,k],bottom=100*bottom,label=long_labels[k],
           width=.67,color=colors[k],hatch=hatches[k],edgecolor='#333333',linewidth=.6)
    for i in range(4):
        val=100*V[i,k]
        if val>=7:
            ax.text(x[i],100*bottom[i]+val/2,f'{val:.0f}%',
                    ha='center',va='center',fontsize=9)
    bottom+=V[:,k]
ax.set_ylim(0,100);ax.set_yticks(np.arange(0,101,20))
ax.set_xticks(x,[g.replace(' — ','\n') for g in GROUPS])
ax.set_ylabel('Estimated viable bacterial population (%)')
ax.set_title('GI population composition — hierarchical shared-donor model')
ax.grid(axis='y',alpha=.18);ax.set_axisbelow(True)
ax.legend(ncol=2,bbox_to_anchor=(.5,-.17),loc='upper center',frameon=False)
fig.text(.5,.01,'Posterior means of animal-normalized fractions (baseline). Day-shared donor means do not force equal D fractions.',
         ha='center',fontsize=8)
fig.tight_layout(rect=[0,.07,1,1])
for ext in ['png','svg']:
    fig.savefig(RESULT_DIR/f'NB15_GI_population_composition_baseline.{ext}',dpi=260,bbox_inches='tight')
plt.show()


In [ ]:
#@title NB15.11 — Credible intervals and donor-prior sensitivity figure
# Main uncertainty figure: 95% credible intervals around each mean animal fraction.
fig,ax=plt.subplots(figsize=(10,6))
for k,cat in enumerate(CATS):
    sub=GROUP_COMP[(GROUP_COMP.scenario=='Baseline') & (GROUP_COMP.component==cat)].set_index('group').loc[GROUPS]
    center=100*sub['median'].to_numpy()
    errlo=100*(sub['median']-sub['q025']).to_numpy()
    errhi=100*(sub['q975']-sub['median']).to_numpy()
    ax.errorbar(np.arange(4)+(k-1.5)*.115,center,yerr=[errlo,errhi],fmt='o',capsize=2,
                color=colors[k],markeredgecolor='#333333',label=long_labels[k])
ax.set_xticks(range(4),[g.replace(' — ','\n') for g in GROUPS])
ax.set_ylim(-2,104)
ax.set_ylabel('Posterior fraction (%) with 95% credible interval')
ax.set_title('GI composition: population-fraction uncertainty')
ax.grid(axis='y',alpha=.18);ax.legend(frameon=False,ncol=2,loc='upper center',bbox_to_anchor=(.5,-.16))
fig.tight_layout()
for ext in ['png','svg']:
    fig.savefig(RESULT_DIR/f'NB15_GI_population_uncertainty.{ext}',dpi=260,bbox_inches='tight')
plt.show()
# Compare the actual four components across all prior choices, not just donor abundance.
fig,ax=plt.subplots(figsize=(12.2,6.3))
scenarios=list(SCENARIOS)
for k,cat in enumerate(CATS):
    means=[]
    for group in GROUPS:
        for scenario in scenarios:
            sub=GROUP_COMP[(GROUP_COMP.scenario==scenario) &
                           (GROUP_COMP.component==cat) &
                           (GROUP_COMP.group==group)]
            assert len(sub)==1
            means.append(float(100*sub['posterior_mean'].iloc[0]))
    # Small gaps separate each of the four sampling groups.
    xloc=np.array([g*5.2+j for g in range(4) for j in range(4)],float)
    ax.plot(xloc+(k-1.5)*.12,means,'o',label=cat,color=colors[k],markersize=5)
ax.set_xticks([i*5.2+1.5 for i in range(4)],[g.replace(' — ','\n') for g in GROUPS])
ax.set_ylim(-2,103)
ax.set_ylabel('Group mean of animal-normalized population fraction (%)')
ax.set_title('GI composition sensitivity to the donor prior')
for i in range(1,4): ax.axvline(i*5.2-.9,color='#aaaaaa',alpha=.4,linewidth=.8)
ax.grid(axis='y',alpha=.15)
ax.legend(ncol=4,frameon=False,loc='upper center',bbox_to_anchor=(.5,-.13))
fig.text(.5,.025,'Within each group, four points per component follow: Baseline, Location18, Spread5, SigmaPrior1.',ha='center',fontsize=8)
fig.tight_layout(rect=[0,.065,1,1])
for ext in ['png','svg']:
    fig.savefig(RESULT_DIR/f'NB15_GI_population_prior_sensitivity.{ext}',dpi=260,bbox_inches='tight')
plt.show()


In [ ]:
#@title NB15.12 — Within-day target versus non-target differences
rows=[]
for scenario in SCENARIOS:
    for day in [10,14]:
        diff=POST_GROUP[scenario,f'Day {day} — Target']-POST_GROUP[scenario,f'Day {day} — Non-target']
        for k,component in enumerate(CATS):
            v=diff[:,k];lo,med,hi=np.quantile(v,[.025,.5,.975])
            rows.append(dict(scenario=scenario,day=day,component=component,
                             difference_median=float(med),q025=float(lo),q975=float(hi),
                             probability_target_lower=float((v<0).mean())))
CONTRASTS=pd.DataFrame(rows)
CONTRASTS.to_csv(RESULT_DIR/'NB15_target_minus_nontarget_fractions.csv',index=False)
display(CONTRASTS[CONTRASTS.scenario=='Baseline'].round(4))
print('Within-day comparisons are observational; they are NOT estimates of conjugation, growth rates or causality.')


In [ ]:
#@title NB15.13 — Posterior predictive checks on four fitted culture plates
PPC_ROWS=[];RESID_ROWS=[]
for scenario,fit in fits.items():
    pred=np.asarray(fit.stan_variable('log_y_rep'))
    assert pred.shape[1:]==(len(data),4)
    for i,rec in data.iterrows():
        for k,plate in enumerate(PLATES):
            y=float(rec[plate]);vec=pred[:,i,k]
            low,med,high=np.quantile(vec,[.025,.5,.975])
            censored=(plate=='N_SmCbTc' and y==0)
            out=(bool(np.log(y)<low or np.log(y)>high) if not censored else np.nan)
            PPC_ROWS.append(dict(scenario=scenario,animal_id=rec.animal_id,
                day=int(rec.day),group=rec.group,plate=plate,observed_CFU_g=y,
                predictive_q025_log=float(low),predictive_median_log=float(med),
                predictive_q975_log=float(high),left_censored=censored,
                outside_predictive_95=out,
                predicted_probability_below_surrogate_LOD=(float((vec<=np.log(T1_LOD[i])).mean())
                    if censored else np.nan)))
            if not censored:
                RESID_ROWS.append(dict(scenario=scenario,group=rec.group,day=int(rec.day),plate=plate,
                     abs_log_residual=abs(float(np.log(y)-med))))
PPC=pd.DataFrame(PPC_ROWS)
RESIDUALS=pd.DataFrame(RESID_ROWS)
PPC.to_csv(RESULT_DIR/'NB15_primary_plate_PPC.csv',index=False)
RESIDUALS.to_csv(RESULT_DIR/'NB15_primary_plate_log_residuals.csv',index=False)
ppc_summary=PPC.groupby(['scenario','plate'],sort=False).agg(
   n=('observed_CFU_g','size'),censored=('left_censored','sum'),outside_95=('outside_predictive_95','sum')).reset_index()
ppc_summary.to_csv(RESULT_DIR/'NB15_PPC_summary.csv',index=False)
display(ppc_summary)
print('Broad in-sample predictive intervals do not establish identification or biological validity.')


In [ ]:
#@title NB15.14 — Check withheld SmRf and SmCb plate counts
HELDOUT_ROWS=[]
for scenario,lp in POST_LOGPOPS.items():
    # These relations assume plate-specific phenotypes remain faithful.
    log_smr=logsumexp(lp[:,:,[0,2,3]],axis=2) # recipients R + T2
    log_smc=logsumexp(lp[:,:,[0,2]],axis=2)   # R + T1
    for i,rec in data.iterrows():
        for plate,loglatent in [('N_SmRf',log_smr[:,i]),('N_SmCb',log_smc[:,i])]:
            y=float(rec[plate]); lo,med,hi=np.quantile(loglatent,[.025,.5,.975])
            HELDOUT_ROWS.append(dict(scenario=scenario,day=int(rec.day),group=rec.group,
                animal_id=rec.animal_id,plate=plate,observed_CFU_g=y,
                latent_q025_CFU_g=np.exp(lo),latent_median_CFU_g=np.exp(med),latent_q975_CFU_g=np.exp(hi),
                outside_latent_95=bool(np.log(max(y,1e-300))<lo or np.log(max(y,1e-300))>hi)))
HELDOUT_DF=pd.DataFrame(HELDOUT_ROWS)
HELDOUT_DF.to_csv(RESULT_DIR/'NB15_heldout_plate_checks.csv',index=False)
display(HELDOUT_DF.groupby(['scenario','plate']).outside_latent_95.agg(['sum','count']))
print('The held-out check compares observed counts with latent credible intervals, which omit plate measurement error.')


In [ ]:
#@title NB15.15 — Summarize conclusions, assumptions and saved outputs
from datetime import datetime, timezone
notes=f'''NB15 — GI hierarchical shared donor (exploratory)
UTC: {datetime.now(timezone.utc).isoformat()}
Source workbook SHA-256: {SOURCE_WORKBOOK_SHA}
Embedded derived GI dataset SHA-256: {DATA_SHA}
32 records; 16 animals sampled twice; target/non-target group sizes 8 each.
Shared donor group-level mean by day is a modelling assumption, not a data-derived test of equality.
Inferred sigma_D is donor-specific between-animal log SD, in addition to fixed within-visit log SD {OCCASION_LOG_SD[1]}.
Other animal scale {NON_D_ANIMAL_LOG_SD}, plate scales {PLATE_LOG_SD.tolist()} and occasion scales {OCCASION_LOG_SD.tolist()} fixed.
Prior sensitivity scenarios: {json.dumps(SCENARIOS,sort_keys=True)}.
Original observed counts are retained, including inconsistent subset counts.
Zero T1 counts are left-censored using unverified surrogate {T1_CFU_ML_SURROGATE_LIMIT} CFU/mL/sample weight.
Four mutually exclusive components R,D,T1,T2-T1 sum to 100% in each posterior draw, and plotting uses group posterior means.
D fractions need not be equal when underlying day mean D is shared across target status.
Fit convergence across scenarios: {diagnostics.to_dict(orient='records')}.
This is not a mechanism, conjugation rate, time series of one bacterial population, or proof of treatment benefit.
Do not alter manuscript or publish KWS data until independently validated and rights checked.
NB05, NB08, NB13, NB14 and manuscripts unchanged.
'''
(RESULT_DIR/'NB15_analysis_summary.txt').write_text(notes)
print(notes)
print('Saved NB15 results under:',RESULT_DIR)
